# Environment Preparation

## Install the `ifran` library

In [1]:
! pip install --upgrade ifran

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 23.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 12.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.4 MB/s  0:00:00
  Attempting uninstall: pydantic_core
    Found existing installation: pydantic_core 2.50.4
    Uninstalling pydantic_core-2.50.4:
      Successfully uninstalled pydantic_core-2.50.4
  Attempting uninstall: pydantic━━━━━━━━━━━━━━━━━━━━━━━━━  4/11 [pydantic_core]
    Found existing installation: pydantic 2.14.3━━━━━━━━━━━━━━  4/11 [pydantic_core]
    Uninstalling pydantic-2.14.3:━━━━━━━━━━━━━━━━━━━━━━━━━  4/11 [pydantic_core]
      Successfully uninstalled pydantic-2.14.3━━━━━━━━━━━━━━━━  4/11 [pydantic_core]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [ifran]m10/11 [ifran]e-core]ings]


In some setup, it might be necessary to install ansible galaxy dependencies as follows.

In [2]:
! ansible-galaxy collection install -r ../../ansible/requirements.yaml

Starting galaxy collection install process
Process install dependency map
Starting collection install process
Installing 'ansible.posix:2.2.2' to '/home/dsaucezi/.ansible/collections/ansible_collections/ansible/posix'
ansible.posix:2.2.2 was installed successfully
Installing 'community.general:14.0.1' to '/home/dsaucezi/.ansible/collections/ansible_collections/community/general'
community.general:14.0.1 was installed successfully
Installing 'kubernetes.core:6.6.0' to '/home/dsaucezi/.ansible/collections/ansible_collections/kubernetes/core'
kubernetes.core:6.6.0 was installed successfully
Installing 'ansible.mysql:5.2.1' to '/home/dsaucezi/.ansible/collections/ansible_collections/ansible/mysql'
ansible.mysql:5.2.1 was installed successfully
Installing 'community.mongodb:1.8.1' to '/home/dsaucezi/.ansible/collections/ansible_collections/community/mongodb'
community.mongodb:1.8.1 was installed successfully
Installing 'community.library_inventory_filtering_v1:1.1.5' to '/home/dsaucezi/.ans

The `ifran` orchestrator is not tied to a specific testbed. It can be used on any infrastructure, as long as the machines:
  - run **Linux**,
  - are reachable over **SSH**, and
  - can run **Kubernetes** (the 5G functions are deployed on a Kubernetes cluster).

In that case, declare the machines with `provisioner: null`, mark them `is_provisioned: true` and give their `ssh_info`. The orchestrator then skips provisioning and goes straight to building the Kubernetes cluster and deploying the network. If you already have a Kubernetes cluster, you can also use it directly, as long as its nodes carry the expected role labels.

The `ifran` orchestrator is also built to integrate seamlessly with the [SLICES-RI](https://doc.slices-ri.eu/) infrastructure, with built-in provisioners (e.g., `bi`, `pos`, `r2lab`) to reserve and set up VMs, bare-metal nodes, radio units and phones for you, so a complete network, including real radio equipment, can be deployed from a single description.

## Integration with SLICES-RI

### Prerequisites

To use the `ifran` orchestrator with SLICES-RI, ensure that you have:
* A valid SLICES account
* A basic understanding of the SLICES CLI

### Step 1 – Register for SLICES

In SLICES, resources are structured into **projects** and **experiments**. Every experiment belongs to a specific project. 

Make sure you have an active SLICES-RI account and are registered with a project. Visit the [SLICES Portal](https://portal.slices-ri.eu/) for details on account creation and project management.

### Step 2 – Install and Configure the SLICES CLI

1. Install the [SLICES Command Line Interface (SLICES CLI)](https://doc.slices-ri.eu/SupportingServices/slicescli.html).
2. Log in using your SLICES credentials.
3. Set your active project context (for example, `post5g-beta`).
4. Install the POS subcommand for the SLICES CLI.

In [3]:
! pip install --upgrade slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Looking in indexes: https://pypi.org/simple, https://doc.slices-ri.eu/pypi/
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24/24 [slices-cli]4 [slices-cli-ai]e]


In [4]:
! slices auth login

Please go to https://portal.slices-ri.eu/oauth/authorize_device?user_code=VNVB-DTBZ and login to continue.
Successfully logged in as 'dsaucezi' 
💾 Saved to /home/dsaucezi/.slices/auth.json
The following projects are available: cnria2026, polars-benchmark, post5g-beta, uclan-cy-playground-1, uoulu-5gtn-ri
👉 Now select the project that you want to use with slices project use <project name>


Select your project with the following command (adapt the project name to yours).

In [5]:
! slices project use post5g-beta

The current project was set to 'post5g-beta', in which you are a lead and which expires on 2026-12-31 23:59 UTC.


If you haven't already done so, [register your public SSH key](https://doc.slices-ri.eu/SupportingServices/slicescli.html#registering-your-public-ssh-key) with SLICES. For example by running:

```bash
ssh-keygen -t ed25519 -N "" -f ~/.ssh/id_ed25519
slices pubkey register ~/.ssh/id_ed25519.pub 
```

These keys are automatically provisioned on all infrastructure resources utilized throughout the `ifran` library.

> **Note:** This initialization and configuration process is required only once when setting up the SLICES CLI.

If you plan on using bare-metal machines (`pos` provisioner), install the POS submodule for the SLICES-CLI:

In [6]:
! pip install --upgrade slices-cli-pos --index-url https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple --extra-index-url https://doc.slices-ri.eu/pypi/

Looking in indexes: https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple, https://doc.slices-ri.eu/pypi/
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 57.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 52.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.4/587.4 kB 29.1 MB/s  0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.70.1
    Uninstalling tqdm-4.70.1:
      Successfully uninstalled tqdm-4.70.1
  Attempting uninstall: setuptools━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/23 [snowballstemmer]
    Found existing installation: setuptools 84.2.1━━━━━━━━━━━━  3/23 [snowballstemmer]
    Uninstalling setuptools-84.2.1:━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/23 [snowballstemmer]
      Successfully uninstalled setuptools-84.2.1━━━━━━━━━━━━━━━━━━  4/23 [setuptools]]
  Attempting uninstall: requestsm━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/23 [setuptoo

You are now fully prepared to run your experiment!